# Governed NL-to-SQL Agent with Query Validation

This example shows how to build a multi-step NL-to-SQL agent using LangGraph where generated SQL must pass a **policy validation gate** before execution. This pattern is essential for production analytics systems where not all generated SQL should be executed directly.

## Architecture

```
User Question
     |
     v
[parse_question] --> [generate_sql] --> [validate_query] --allow--> [execute_query]
                                              |                          |
                                           deny/review                   v
                                              |                    [format_result]
                                              v
                                       [reject_query]
```

Key features:
- **Policy engine** with deny/review/allow decisions
- **Audit trail** tracking every step
- **Conditional routing** based on validation results

In [ ]:
%%capture --no-stderr
%pip install -U langgraph langchain-openai

In [ ]:
import os
import re
import sqlite3
from datetime import datetime
from typing import Annotated, Literal, TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages

## Define State

The state tracks the question, generated SQL, policy decision, query results, and a full audit trail.

In [ ]:
class AgentState(TypedDict):
    question: str
    sql: str
    policy_decision: Literal["allow", "deny", "review"]
    policy_reason: str
    results: list[dict]
    error: str
    audit_trail: list[dict]

## Set Up Demo Database

In [ ]:
def setup_demo_db() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:")
    conn.execute(
        "CREATE TABLE sales (id INTEGER PRIMARY KEY, region TEXT, "
        "product TEXT, amount REAL, sale_date TEXT)"
    )
    rows = [
        (1, "APAC", "Widget", 1200.0, "2026-01-15"),
        (2, "EMEA", "Gadget", 850.0, "2026-01-20"),
        (3, "APAC", "Widget", 2300.0, "2026-02-01"),
        (4, "NA", "Gadget", 1750.0, "2026-02-10"),
        (5, "EMEA", "Widget", 900.0, "2026-03-01"),
    ]
    conn.executemany("INSERT INTO sales VALUES (?,?,?,?,?)", rows)
    conn.commit()
    return conn

db = setup_demo_db()

## Define Policy Engine

The policy engine inspects generated SQL and decides whether to **allow**, **deny**, or flag for **review**.

In [ ]:
DANGEROUS_PATTERNS = [
    (r"\bDROP\b", "DROP statements are not allowed"),
    (r"\bDELETE\b", "DELETE statements are not allowed"),
    (r"\bINSERT\b", "INSERT statements are not allowed"),
    (r"\bUPDATE\b", "UPDATE statements are not allowed"),
    (r"\bALTER\b", "ALTER statements are not allowed"),
    (r"\bTRUNCATE\b", "TRUNCATE statements are not allowed"),
]

REVIEW_PATTERNS = [
    (r"SELECT\s+\*", "SELECT * queries should specify columns explicitly"),
    (r"\bJOIN\b.*\bJOIN\b", "Multi-join queries require review"),
]


def check_policy(sql: str) -> tuple[str, str]:
    """Returns (decision, reason) where decision is allow/deny/review."""
    sql_upper = sql.upper()
    for pattern, reason in DANGEROUS_PATTERNS:
        if re.search(pattern, sql_upper):
            return "deny", reason
    for pattern, reason in REVIEW_PATTERNS:
        if re.search(pattern, sql_upper):
            return "review", reason
    return "allow", "Query passed all policy checks"

## Define Graph Nodes

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def generate_sql(state: AgentState) -> AgentState:
    """Translate natural language question to SQL."""
    prompt = (
        f"Given a table `sales` with columns: id, region, product, amount, sale_date\n"
        f"Write a SQL SELECT query for: {state['question']}\n"
        f"Return ONLY the SQL query, no explanation."
    )
    response = llm.invoke(prompt)
    sql = response.content.strip().strip("`").removeprefix("sql").strip()
    audit_entry = {
        "step": "generate_sql",
        "timestamp": datetime.now().isoformat(),
        "input": state["question"],
        "output": sql,
    }
    return {**state, "sql": sql, "audit_trail": state.get("audit_trail", []) + [audit_entry]}


def validate_query(state: AgentState) -> AgentState:
    """Run the generated SQL through the policy engine."""
    decision, reason = check_policy(state["sql"])
    audit_entry = {
        "step": "validate_query",
        "timestamp": datetime.now().isoformat(),
        "decision": decision,
        "reason": reason,
    }
    return {
        **state,
        "policy_decision": decision,
        "policy_reason": reason,
        "audit_trail": state.get("audit_trail", []) + [audit_entry],
    }


def execute_query(state: AgentState) -> AgentState:
    """Execute the validated SQL against the database."""
    try:
        cursor = db.execute(state["sql"])
        columns = [desc[0] for desc in cursor.description]
        rows = [dict(zip(columns, row)) for row in cursor.fetchall()]
        audit_entry = {
            "step": "execute_query",
            "timestamp": datetime.now().isoformat(),
            "row_count": len(rows),
        }
        return {**state, "results": rows, "audit_trail": state.get("audit_trail", []) + [audit_entry]}
    except Exception as e:
        return {**state, "error": str(e), "results": []}


def reject_query(state: AgentState) -> AgentState:
    """Handle rejected or review-flagged queries."""
    audit_entry = {
        "step": "reject_query",
        "timestamp": datetime.now().isoformat(),
        "decision": state["policy_decision"],
        "reason": state["policy_reason"],
    }
    return {
        **state,
        "results": [],
        "error": f"Query {state['policy_decision']}: {state['policy_reason']}",
        "audit_trail": state.get("audit_trail", []) + [audit_entry],
    }

## Build the Graph

The conditional edge after `validate_query` routes to either `execute_query` (allowed) or `reject_query` (denied/review).

In [ ]:
def route_after_validation(state: AgentState) -> str:
    if state["policy_decision"] == "allow":
        return "execute_query"
    return "reject_query"


workflow = StateGraph(AgentState)

workflow.add_node("generate_sql", generate_sql)
workflow.add_node("validate_query", validate_query)
workflow.add_node("execute_query", execute_query)
workflow.add_node("reject_query", reject_query)

workflow.set_entry_point("generate_sql")
workflow.add_edge("generate_sql", "validate_query")
workflow.add_conditional_edges("validate_query", route_after_validation)
workflow.add_edge("execute_query", END)
workflow.add_edge("reject_query", END)

app = workflow.compile()

## Run: Safe Query (Allowed)

In [ ]:
result = app.invoke({"question": "Show total sales amount by region"})

print(f"SQL: {result['sql']}")
print(f"Policy: {result['policy_decision']} - {result['policy_reason']}")
print(f"Results: {result['results']}")
print(f"\nAudit trail ({len(result['audit_trail'])} entries):")
for entry in result["audit_trail"]:
    print(f"  [{entry['step']}] {entry.get('decision', entry.get('output', entry.get('row_count', '')))}")

## Run: Dangerous Query (Denied)

In [ ]:
result = app.invoke({"question": "Delete all sales records from EMEA region"})

print(f"SQL: {result['sql']}")
print(f"Policy: {result['policy_decision']} - {result['policy_reason']}")
print(f"Error: {result.get('error', 'None')}")
print(f"\nAudit trail ({len(result['audit_trail'])} entries):")
for entry in result["audit_trail"]:
    print(f"  [{entry['step']}] {entry.get('decision', entry.get('output', ''))}")